# Linear Regression: Pune food delivery orders

A food delivery app in Pune shows every customer a delivery time when they place an order. Right now the app shows the same **"35 minutes"** for everyone, so people who live close get their food early, and people who live far get it late and leave angry reviews.

The operations team has the records of 200 past orders. They want a model that looks at a new order and says **how many minutes it will take**, so the app can show an honest time.

**Goal:** Predict delivery_minutes (a number) for a new order. Predicting a number is a regression problem, so we start with Linear Regression.

## The data

200 past orders, one row per order. The file is `/data/deliveries.csv`.

- `order_id`: Order number (just an id, not used by the model)
- `distance_km`: Distance from the restaurant to the customer
- `prep_minutes`: Minutes the restaurant took to pack the food
- `peak_hour`: Was it lunch or dinner rush? (Yes / No)
- `raining`: Was it raining? (Yes / No)
- `delivery_minutes`: What we want to predict: total minutes until delivery

## 1. Load the data and take a first look

Every ML project starts by looking at the data. How many rows are there, what do the columns look like, and is anything missing?

In [ ]:
import pandas as pd

df = pd.read_csv("data/deliveries.csv")
print("Rows and columns:", df.shape)
print("Missing values:", df.isna().sum().sum())
df.head()

## 2. Look for a pattern

Linear Regression draws a **straight line** through the data. So first we check: does delivery time go up in a straight-line way when the distance goes up? A scatter plot answers this in one look.

In [ ]:
import matplotlib.pyplot as plt

plt.scatter(df["distance_km"], df["delivery_minutes"], alpha=0.6)
plt.xlabel("Distance (km)")
plt.ylabel("Delivery time (minutes)")
plt.title("Longer distance, longer delivery")
plt.show()

## 3. Split the data: one part to learn, one part to test

We hide 20% of the orders from the model. The model learns from the other 80%, and later we test it on the hidden orders, just like a real exam with questions the student has not seen.

- **X** is what the model looks at (the input). Here, only distance.
- **y** is the answer we want it to learn (delivery_minutes).

In [ ]:
from sklearn.model_selection import train_test_split

X = df[["distance_km"]]
y = df["delivery_minutes"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print("Orders to learn from:", len(X_train))
print("Orders kept for the test:", len(X_test))

## 4. Train the model

`fit` is where the learning happens. The model finds the best straight line: **minutes = base time + (minutes per km × distance)**.

The two numbers it learns have a plain business meaning, which is what you explain to the manager.

In [ ]:
from sklearn.linear_model import LinearRegression

model = LinearRegression()
model.fit(X_train, y_train)

per_km = model.coef_[0]
base = model.intercept_
print(f"Every extra km adds about {per_km:.1f} minutes")
print(f"Base time (pickup, packing, parking): about {base:.1f} minutes")

## 5. Test it on orders it has never seen

Now the exam. We predict the hidden orders and compare with what really happened.

- **MAE** (mean absolute error): on average, how many minutes the prediction is off by. Smaller is better.
- **R²**: how much of the ups and downs in delivery time the model explains, from 0 to 1. Closer to 1 is better.

If R² looks low, that is the real lesson: distance alone does not explain delivery time well. The next step fixes it.

In [ ]:
from sklearn.metrics import mean_absolute_error, r2_score

pred = model.predict(X_test)
print(f"MAE: {mean_absolute_error(y_test, pred):.1f} minutes")
print(f"R²:  {r2_score(y_test, pred):.2f}")

plt.scatter(X_test["distance_km"], y_test, label="Real time", alpha=0.7)
line_x = pd.DataFrame({"distance_km": [X["distance_km"].min(), X["distance_km"].max()]})
plt.plot(line_x["distance_km"], model.predict(line_x), color="red", linewidth=2, label="Model's line")
plt.xlabel("Distance (km)")
plt.ylabel("Delivery time (minutes)")
plt.legend()
plt.show()

## 6. Give the model more information

Distance alone misses a lot: a rainy dinner rush is slower than a sunny afternoon. Let's add the other columns. This is **multiple linear regression**: one line, many inputs.

A model only understands numbers, so first we turn **Yes / No** into **1 / 0**.

In [ ]:
df["peak"] = (df["peak_hour"] == "Yes").astype(int)
df["rain"] = (df["raining"] == "Yes").astype(int)

features = ["distance_km", "prep_minutes", "peak", "rain"]
X = df[features]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model2 = LinearRegression()
model2.fit(X_train, y_train)
pred2 = model2.predict(X_test)

print(f"Distance only   → MAE {mean_absolute_error(y_test, pred):.1f} min, R² {r2_score(y_test, pred):.2f}")
print(f"All four inputs → MAE {mean_absolute_error(y_test, pred2):.1f} min, R² {r2_score(y_test, pred2):.2f}")

## 7. What did the model learn?

Each input gets its own number (a coefficient). It says how many minutes that input adds, keeping everything else the same. This table is the part a manager cares about most.

In [ ]:
effects = pd.DataFrame({"input": features, "adds_minutes": model2.coef_.round(1)})
effects

## 8. Predict a new order

The model is ready to use. A customer 6 km away orders during the dinner rush while it is raining, and the restaurant needs 15 minutes. **Change the numbers and run again** to see how the time changes.

In [ ]:
new_order = pd.DataFrame({
    "distance_km": [6.0],
    "prep_minutes": [15],
    "peak": [1],   # 1 = Yes, 0 = No
    "rain": [1],   # 1 = Yes, 0 = No
})
eta = model2.predict(new_order)[0]
print(f"Show the customer: about {round(eta)} minutes")

## What I learned

- Linear Regression predicts a number by fitting a straight line (or a flat plane, with many inputs).
- Always test on data the model has not seen: train_test_split.
- MAE tells the error in the same unit as the answer (minutes here), so anyone can understand it.
- Coefficients explain the model: each one says how much that input adds.
- Text columns must become numbers before training.

## When not to use Linear Regression

- When the pattern is a curve, not a straight line (for example, prices that jump after a point).
- When the answer is a category like Yes / No (will the customer cancel?). Use Logistic Regression for that.
- When there are big outliers: one 3-hour delivery can pull the whole line.